# Perfilado local (Fase P0)
Perfila un archivo crudo en DuckDB y muestra el **Data Profile**: lo único que viajaría al LLM.

In [ ]:
%load_ext autoreload
%autoreload 2
import json
import pandas as pd
from dfir_copilot.profiling import LogProfiler

RUTA = "/workspace/data/raw/three_months.csv"   # cambia por cualquier CSV, TSV, JSON/NDJSON o Parquet
IDIOMA = "es"                                    # "es" o "en"

perfil = LogProfiler(RUTA, lang=IDIOMA).profile()
carga = perfil.to_llm_json()
print(perfil.source.format, perfil.source.encoding, repr(perfil.source.delimiter), '| filas:', perfil.dataset.row_count)
print('tamaño del perfil:', len(carga.encode('utf-8')), 'bytes')

In [ ]:
display(pd.DataFrame([{'canónico': m.canonical, 'campo': m.field, 'puntuación': m.score, 'método': m.method}
                      for m in perfil.mapping]))
print('Sin mapear:', perfil.unmapped_fields, '| Ambiguos:', perfil.ambiguous)
print('Tipo de log:', [(h.label, h.score) for h in perfil.log_type_hints])
if perfil.timestamp:
    print('Fecha:', perfil.timestamp.model_dump())

In [ ]:
for w in perfil.warnings:
    print(f'[{w.code}] {w.message}')

In [ ]:
# Lo que recibiría el LLM, completo
print(json.dumps(json.loads(carga), indent=2, ensure_ascii=False))